In [1]:
import pandas as pd
import numpy as np

df_freq = pd.read_csv("../../data/freMTPL2freq.csv")
df_sev = pd.read_csv("../../data/df_sev_clean.csv")

# ۱. ادغام دو جدول
claims_per_policy = df_sev.groupby('IDpol')['ClaimAmount'].sum().reset_index()
df_merged = pd.merge(df_freq, claims_per_policy, on='IDpol', how='left')
df_merged['ClaimAmount'] = df_merged['ClaimAmount'].fillna(0)

# ۲. شناسایی مبالغ خسارت بسیار سنگین (Extreme Claim Amounts)
# بر اساس صدک ۹۹.۹ یا روش چارکی (IQR)
q3 = df_sev[df_sev['ClaimAmount'] > 0]['ClaimAmount'].quantile(0.75)
q1 = df_sev[df_sev['ClaimAmount'] > 0]['ClaimAmount'].quantile(0.25)
iqr = q3 - q1

extreme_threshold = q3 + 3 * iqr

extreme_claims = df_merged[df_merged['ClaimAmount'] > extreme_threshold].sort_values(
    by='ClaimAmount', ascending=False
)

# ۳. شناسایی ترکیبات غیرعادی بخش‌ها (Unusual Combinations)
# مانند: فرکانس ادعای بسیار بالا در پوشش‌های با مدت زمان خیلی کم (Low Exposure)
unusual_exposure = df_merged[(df_merged['Exposure'] < 0.1) & (df_merged['ClaimNb'] >= 2)]

# یا خودروهای ارزان/مسن با خسارت‌های بسیار بالا
unusual_vehicle_claims = df_merged[(df_merged['VehAge'] > 15) & (df_merged['ClaimAmount'] > 50000)]

print(f"Number of Extreme Outliers (> {extreme_threshold:.2f}):", len(extreme_claims))
print("\nTop 5 Extreme Claims:")
print(extreme_claims[['IDpol', 'ClaimNb', 'Exposure', 'DrivAge', 'VehBrand', 'ClaimAmount']].head())

print("\nUnusual High Frequency with Low Exposure:")
print(unusual_exposure[['IDpol', 'Exposure', 'ClaimNb', 'ClaimAmount']].head())

# تحلیل Outlierها نشان می‌دهد که بخشی از بیمه‌نامه‌ها دارای
# خسارت‌های بسیار سنگین هستند و برخی نیز ترکیب‌های غیرعادی دارند.

# آستانه شناسایی خسارت‌های بسیار سنگین برابر با 2791.56 است
# و در مجموع 2695 بیمه‌نامه بالاتر از این آستانه قرار گرفته‌اند.

# بزرگ‌ترین مورد مربوط به بیمه‌نامه 1120377 است:
# مبلغ خسارت ≈ 4,075,400
# تعداد خسارت = 1
# Exposure = 0.22
# سن راننده = 19
# برند خودرو = B2

# همچنین بیمه‌نامه 110846 با 2 خسارت،
# مجموع خسارت ≈ 1,404,185 داشته است.

# در بخش Exposure پایین نیز موارد غیرعادی مشاهده شد؛
# برای مثال بیمه‌نامه 80559 با Exposure برابر 0.07،
# دارای 2 خسارت و مجموع خسارت ≈ 1,904 است.

# بیمه‌نامه 1015603 نیز با Exposure برابر 0.04،
# دارای 2 خسارت و مجموع خسارت ≈ 6,790 است.

# این موارد الزاماً خطا یا تقلب نیستند،
# اما به‌عنوان موارد غیرعادی (Unusual Cases) می‌توانند
# برای بررسی دقیق‌تر و تحلیل ریسک مورد توجه قرار گیرند.

# نکته فنی:
# آستانه Extreme در سطح خسارت‌های df_sev محاسبه شده،
# اما روی ClaimAmount تجمیع‌شده در سطح بیمه‌نامه اعمال شده است.
# بنابراین برای یک تحلیل کاملاً دقیق، بهتر است آستانه نیز
# در همان سطح بیمه‌نامه محاسبه شود.

Number of Extreme Outliers (> 2791.56): 2695

Top 5 Extreme Claims:
            IDpol  ClaimNb  Exposure  DrivAge VehBrand  ClaimAmount
150015  1120377.0        1      0.22       19       B2   4075400.56
54315    110846.0        2      0.43       20       B1   1404185.52
270605  2141337.0        1      0.32       18       B2   1301172.60
416613  3122016.0        1      0.91       40      B11    774411.50
203383  2008127.0        3      0.36       57       B4    399213.66

Unusual High Frequency with Low Exposure:
           IDpol  Exposure  ClaimNb  ClaimAmount
39844    80559.0      0.07        2      1904.46
46541    94073.0      0.09        2      1243.46
55922   115165.0      0.04        2      2154.87
94251  1012279.0      0.07        2      6893.26
97575  1015603.0      0.04        2      6790.36
